In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import pandas as pd
from PopSynthesis.Methods.IPSF.const import (
    data_dir,
    processed_dir,
    POOL_SIZE,
)
from PopSynthesis.Methods.IPSF.utils.pool_utils import create_pool

/Users/bobla/Documents/repos/PopSyn_Monash/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy
import scipy
import statsmodels.api as sm

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Statsmodels:", sm.__version__)

NumPy: 2.4.6
SciPy: 1.15.3
Statsmodels: 0.14.4


In [4]:
from pathlib import Path

def inspect_path(path):
    p = Path(path)

    if not p.exists():
        print(f"Path does not exist: {p}")
        return

    if p.is_file():
        print(f"File exists: {p}")
    elif p.is_dir():
        print(f"Directory exists: {p}")
        print("Contents:")
        for item in sorted(p.iterdir()):
            print(f"  {'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

In [5]:
inspect_path(processed_dir)

Path does not exist: /Users/bobla/Documents/repos/PopSyn_Monash/PopSynthesis/Methods/IPSF/data/processed


In [6]:
test_path = Path("/Users/bobla/Documents/repos/PopSyn_Monash/IO/processed_data")
inspect_path(test_path)

Directory exists: /Users/bobla/Documents/repos/PopSyn_Monash/IO/processed_data
Contents:
  [FILE] HH_pool.csv
  [FILE] HH_pool_condensed.csv
  [FILE] census_POA.csv
  [FILE] census_POA_with_nil_income.csv
  [FILE] ori_sample_hh.csv
  [DIR] output


In [7]:
seed = pd.read_csv(test_path / "ori_sample_hh.csv")

In [8]:
seed

,hhid,dwelltype,owndwell,hhinc,totalvehs,hhsize,Spouse,Child,Others,Grandchild,_weight
0,Y12H0000101,Separate House,Fully Owned,1000-1249,2,4,1,2,0,0,92.33
1,Y12H0000102,Separate House,Being Purchased,1500-1749,3,4,1,2,0,0,90.31
2,Y12H0000103,Separate House,Being Purchased,1000-1249,2,4,1,2,0,0,91.63
3,Y12H0000104,Separate House,Fully Owned,1750-1999,3,3,1,1,0,0,91.78
4,Y12H0000107,Separate House,Being Purchased,1750-1999,1,3,0,2,0,0,270.81
...,...,...,...,...,...,...,...,...,...,...,...
30798,Y20H4001024,Separate House,Being Purchased,Nil income,3,2,1,0,0,0,300.87
30799,Y20H4001028,Separate House,Fully Owned,500-649,2,2,1,0,0,0,858.94
30800,Y20H4001029,Separate House,Fully Owned,650-799,2,2,1,0,0,0,303.17
30801,Y20H4001036,Separate House,Fully Owned,400-499,1,1,0,0,0,0,303.17


In [9]:
# Keep these if you want the BN to preserve their correlation with household traits.
attributes = [
    "dwelltype",
    "owndwell",
    "hhinc",
    "totalvehs",
    "hhsize",
    # "Spouse",
    # "Child",
    # "Others",
    # "Grandchild",
]

# hhid is a unique identifier; never learn/sample it.
seed = seed[attributes + ["_weight"]].copy()

# SAA treats pool states as categorical strings. Keep the BN state labels aligned.
seed[attributes] = seed[attributes].astype(str)

state_names = {
    attribute: sorted(seed[attribute].dropna().unique().tolist())
    for attribute in attributes
}

pool = create_pool(
    seed=seed,
    state_names=state_names,
    pool_sz=1_000_000,  # begin here; use 5_000_000 only if needed
    special=False,
)

# pool_file.parent.mkdir(parents=True, exist_ok=True)
# pool.to_csv(pool_file, index=False)

# print(f"Wrote {len(pool):,} records to {pool_file}")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C'}


Learn BN


  0%|          | 5/1000000 [00:00<2:27:14, 113.20it/s]
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'dwelltype': 'C', 'owndwell': 'C', 'hhinc': 'C', 'totalvehs': 'C', 'hhsize': 'C', '_weight': 'N'}


Doing the sampling


Generating for node: owndwell: 100%|██████████| 5/5 [00:01<00:00,  4.01it/s]


In [16]:
pool

,dwelltype,owndwell,hhinc,totalvehs,hhsize
0,Flat or Apartment,Being Rented,1750-1999,1,1
1,Other,Fully Owned,2500-2999,1,1
2,Separate House,Being Rented,1750-1999,2,4
3,Separate House,Fully Owned,400-499,2,5
4,Separate House,Being Purchased,650-799,2,2
...,...,...,...,...,...
999995,Separate House,Being Purchased,1250-1499,3,3
999996,Separate House,Being Purchased,3000-3499,2,2
999997,Separate House,Fully Owned,1750-1999,1,2
999998,Terrace/Townhouse,Being Rented,1000-1249,1,1


In [18]:
pool.to_csv(test_path / "HH_pool.csv", index=False)

In [ ]:
import polars as pl

# Each distinct attribute combination becomes one pool record.
# Do not include an ID or count column in the grouping keys.
record_columns = pool.columns.tolist()

condensed_pool = (
    pl.from_pandas(pool)
    # Match SAA's current categorical/string treatment.
    .with_columns(pl.col(record_columns).cast(pl.String))
    .group_by(record_columns)
    .agg(pl.len().alias("count"))
    # Sorting makes pool_id deterministic for a given pool.
    .sort(record_columns)
    .with_row_index("pool_id")
    .select(["pool_id", *record_columns, "count"])
)

assert condensed_pool["count"].sum() == len(pool)
assert condensed_pool.select(pl.struct(record_columns).n_unique()).item() == len(
    condensed_pool
)

condensed_pool

pool_id,dwelltype,owndwell,hhinc,totalvehs,hhsize,count
u32,str,str,str,str,str,u32
0,"""Flat or Apartment""","""Being Purchased""","""1-149""","""0""","""1""",64
1,"""Flat or Apartment""","""Being Purchased""","""1-149""","""0""","""2""",12
2,"""Flat or Apartment""","""Being Purchased""","""1-149""","""0""","""3""",1
3,"""Flat or Apartment""","""Being Purchased""","""1-149""","""0""","""4""",2
4,"""Flat or Apartment""","""Being Purchased""","""1-149""","""1""","""1""",124
…,…,…,…,…,…,…
8445,"""Terrace/Townhouse""","""Something Else""","""Nil income""","""2""","""2""",19
8446,"""Terrace/Townhouse""","""Something Else""","""Nil income""","""2""","""3""",2
8447,"""Terrace/Townhouse""","""Something Else""","""Nil income""","""2""","""4""",3


In [12]:
condensed_pool["count"].sum()

1000000

In [13]:
condensed_pool.write_csv(test_path / "HH_pool_condensed.csv")

In [10]:
# pool = pd.read_csv(test_path / "HH_pool_condensed.csv")

print(sorted(pool["hhinc"].astype(str).unique()))
print(sorted(pool["hhsize"].astype(str).unique()))
print(sorted(pool["totalvehs"].astype(str).unique()))

['1-149', '1000-1249', '1250-1499', '150-299', '1500-1749', '1750-1999', '2000-2499', '2500-2999', '300-399', '3000-3499', '3500-3999', '400-499', '4000-4499', '4500-4999', '500-649', '5000-5999', '6000-7999', '650-799', '800-999', '8000+', 'Nil income']
['1', '2', '3', '4', '5', '6', '7', '8+']
['0', '1', '2', '3', '4+']


In [26]:
census_file = test_path / "census_POA.csv"
pool_file = test_path / "HH_pool.csv" # the full, uncondensed BN pool
output_file = test_path / "hh_syn_saa.csv"

a = pd.read_csv(census_file)
[x.split("_")[-1] for x in a.columns if "hhinc_" in x]

['1-149',
 '1000-1249',
 '1250-1499',
 '150-299',
 '1500-1749',
 '1750-1999',
 '2000-2499',
 '2500-2999',
 '300-399',
 '3000-3499',
 '3500-3999',
 '400-499',
 '4000-4499',
 '4500-4999',
 '500-649',
 '5000-5999',
 '6000-7999',
 '650-799',
 '800-999',
 '8000+',
 'Negative income']

In [13]:
from PopSynthesis.Methods.IPSF.run import run_saa

census_file = test_path / "census_POA_hh_saa.csv"
pool_file = test_path / "HH_pool.csv" # the full, uncondensed BN pool
output_file = test_path / "hh_syn_saa.csv"

output_file.parent.mkdir(parents=True, exist_ok=True)

run_saa(
    marg_file=census_file,
    pool_file=pool_file,
    zone_field="zone_id",
    output_file=output_file,

    # Columns retained in the final synthetic households.
    considered_atts=[
        "hhsize",
        "hhinc",
        "totalvehs",
        "dwelltype",
        "owndwell",
    ],

    # Census-controlled attributes.
    ordered_to_adjust_atts=[
        "hhsize",
        "totalvehs",
        "hhinc",
        "dwelltype",
        "owndwell",
    ],

    # Number of sequential loopback passes, not wall-clock time.
    max_run_time=2,

    # Randomly discard 30% before keeping the non-oversupplied population
    # between passes. Use 0 for an initial debugging run.
    extra_rm_frac=0.3,

    # Final-pass priority.
    last_adjustment_order=[
        "hhsize",
        "totalvehs",
        "hhinc",
        "dwelltype",
        "owndwell",
    ],

    # Useful while checking what happens at each adjustment.
    output_each_step=False,
    add_name_for_step_output="bn_pool_test",

    # Keep False initially; zero-cell handling is still experimental.
    include_zero_cell_values=False,

    # Store each adjustment/pass here.
    meta_output_dir=Path("IO/output/meta"),
)

These atts are synthesized but not adjusted: []
For run 0, order is: ['hhinc', 'hhsize', 'dwelltype', 'totalvehs', 'owndwell'], aim for 2420221.0 agents
ADJUSTING FOR hhinc
ADJUSTING FOR hhsize
DOING zone 3996
ADJUSTING FOR dwelltype
DOING zone 3996
ADJUSTING FOR totalvehs
DOING zone 3996
ADJUSTING FOR owndwell
DOING zone 3996
removed first 726066 agents
DOING deleting to match cencus diff for 3996
For run 1, order is: ['hhsize', 'totalvehs', 'hhinc', 'dwelltype', 'owndwell'], aim for 726982 agents
ADJUSTING FOR hhsize
ADJUSTING FOR totalvehs
DOING zone 3996
ADJUSTING FOR hhinc
DOING zone 3996
ADJUSTING FOR dwelltype
DOING zone 3996
ADJUSTING FOR owndwell
DOING zone 3996


In [12]:
import pandas as pd
from pathlib import Path

raw = pd.read_csv(census_file)

atts = ["hhsize", "dwelltype", "hhinc", "totalvehs", "owndwell"]

out = pd.DataFrame({
    ("zone_id", ""): raw["POA"].astype(str),
})

for att in atts:
    for col in raw.columns:
        prefix = f"{att}_"
        if col.startswith(prefix):
            state = col.removeprefix(prefix)
            out[(att, state)] = raw[col]

out.columns = pd.MultiIndex.from_tuples(out.columns)

census_file = test_path / "census_POA_saa.csv"
out.to_csv(census_file, index=False)

In [34]:
from pathlib import Path
import pandas as pd

census_file = test_path / "census_POA.csv"
output_dir = test_path / "output/census_checks"
output_dir.mkdir(parents=True, exist_ok=True)

census = pd.read_csv(census_file)

groups = {
    "households": {
        "total_col": "tot_hh",
        "attributes": [
            "hhsize",
            "dwelltype",
            "hhinc",
            "totalvehs",
            "owndwell",
        ],
    },
    "persons": {
        "total_col": "tot_pp",
        "attributes": [
            "age",
            "persinc",
            "sex",
            "anywork",
        ],
    },
}

summary = []
all_zone_checks = []

for level, config in groups.items():
    total_col = config["total_col"]

    for att in config["attributes"]:
        state_cols = [col for col in census.columns if col.startswith(f"{att}_")]

        if not state_cols:
            print(f"SKIP: no columns found for {att}")
            continue

        marginal_sum = census[state_cols].sum(axis=1)
        residual = census[total_col] - marginal_sum

        # hhinc fallback: interpret residual as Nil income.
        # This does NOT modify the census file.
        if att == "hhinc":
            checked_sum = marginal_sum + residual
        else:
            checked_sum = marginal_sum

        is_match = checked_sum == census[total_col]

        check = pd.DataFrame({
            "zone": census["POA"],
            "level": level,
            "attribute": att,
            "target_total": census[total_col],
            "marginal_sum": marginal_sum,
            "residual": residual,
            "matches_target": is_match,
        })

        all_zone_checks.append(check)

        summary.append({
            "level": level,
            "attribute": att,
            "n_states": len(state_cols),
            "n_zones": len(census),
            "n_matching": int(is_match.sum()),
            "n_not_matching": int((~is_match).sum()),
            "total_residual": int(residual.sum()),
            "min_residual": int(residual.min()),
            "max_residual": int(residual.max()),
        })

summary_df = pd.DataFrame(summary)
zone_checks_df = pd.concat(all_zone_checks, ignore_index=True)

# summary_df.to_csv(output_dir / "marginal_total_summary.csv", index=False)
# zone_checks_df.to_csv(output_dir / "marginal_total_by_zone.csv", index=False)

display(summary_df)

# All failures, excluding hhinc because it uses the Nil-income fallback above.
failures = zone_checks_df[
    (~zone_checks_df["matches_target"])
    & (zone_checks_df["attribute"] != "hhinc")
]

display(failures)
print(f"Non-hhinc failures: {len(failures):,}")

,level,attribute,n_states,n_zones,n_matching,n_not_matching,total_residual,min_residual,max_residual
0,households,hhsize,8,691,691,0,0,0,0
1,households,dwelltype,5,691,691,0,0,0,0
2,households,hhinc,21,691,691,0,49418,0,1916
3,households,totalvehs,5,691,691,0,0,0,0
4,households,owndwell,4,691,691,0,0,0,0
5,persons,age,11,691,691,0,0,0,0
6,persons,persinc,13,691,691,0,0,0,0
7,persons,sex,2,691,691,0,0,0,0
8,persons,anywork,2,691,691,0,0,0,0


,zone,level,attribute,target_total,marginal_sum,residual,matches_target


Non-hhinc failures: 0
